# Code 8.1.1: Shift by one, loss, and perplexity with GPT-2

Cuts the example sentence into inputs and labels shifted by one token, computes the loss of the released GPT-2 small model by hand and with Hugging Face's built-in loss, and checks that an untrained GPT-2 starts near $`\ln V`$.


In [ ]:
import math
import tiktoken
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Config

enc = tiktoken.get_encoding("gpt2")
text = "The cat sat on the mat because it was tired."
ids = enc.encode(text)
print(len(ids), enc.n_vocab)                           # tokens in the text, vocabulary size V
x = torch.tensor(ids)
inputs, labels = x[:-1], x[1:]                         # a block of n + 1 tokens gives n training positions
for i, l in zip(inputs[:4].tolist(), labels[:4].tolist()):
    print(repr(enc.decode([i])), "->", repr(enc.decode([l])))

model = GPT2LMHeadModel.from_pretrained("gpt2").eval()
with torch.no_grad():
    logits = model(inputs[None]).logits[0]             # (n, V): one distribution per position
    loss = F.cross_entropy(logits, labels)             # average of -log p(x_{t+1} | x_{<=t})
    hf_loss = model(x[None], labels=x[None]).loss      # Hugging Face shifts the labels internally
print(f"loss {loss:.4f} nats, HF loss {hf_loss:.4f}, perplexity {loss.exp():.1f}")

# An untrained model spreads its probability almost evenly: loss close to ln V
torch.manual_seed(0)
untrained = GPT2LMHeadModel(GPT2Config()).eval()
with torch.no_grad():
    print(f"untrained loss {untrained(x[None], labels=x[None]).loss:.3f}, ln V = {math.log(enc.n_vocab):.3f}")
# 11 50257
# 'The' -> ' cat'
# ' cat' -> ' sat'
# ' sat' -> ' on'
# ' on' -> ' the'
# loss 3.8762 nats, HF loss 3.8762, perplexity 48.2
# untrained loss 11.048, ln V = 10.825
